# Taller N°1 — Consumo de APIs

**Asignatura:** Machine Learning — MLY1101 (Sección 001V)  
**Estudiante:** Hernán Lippke  
**Objetivo:** Extraer datos automáticamente desde 3 APIs públicas, generar 3 archivos de datos independientes y publicar el trabajo en GitHub.

> **Reproducibilidad:** el notebook está pensado para ejecutarse con *Entorno de ejecución → Ejecutar todas* y regenerar los 3 datasets sin intervención manual. Las 3 APIs son **sin API key**.

## 1. Pregunta u objetivo común

> **¿Qué información puede recopilarse sobre los países del mundo desde fuentes geográfico-demográficas, económicas y culturales?**

Cada fuente aporta a esta pregunta desde un ángulo distinto.  
**No se busca responder la pregunta ni integrar, unir o cruzar las fuentes.** Cada API se consume y se guarda de forma independiente.

## 2. Fuentes seleccionadas

Tres APIs públicas de la lista `public-apis/public-apis`, todas **sin API key**:

| # | API | Aporta a la pregunta | Enlace |
|---|-----|----------------------|--------|
| 1 | **REST Countries** | Geografía y demografía (capital, región, población, área, idiomas, monedas, coordenadas) | https://restcountries.com/ |
| 2 | **World Bank** | Economía (indicador PIB per cápita por país) | https://data.worldbank.org/ |
| 3 | **Nager.Date** | Cultura (días festivos oficiales por país) | https://date.nager.at/ |

## 3. Configuración común

Imports, carpeta de salida y utilidades compartidas por las 3 descargas.

In [1]:
import os, time, json
import requests
import pandas as pd

OUT_DIR = "."            # misma carpeta del repositorio
MIN_RECORDS = 200        # objetivo mínimo de registros por API
os.makedirs(OUT_DIR, exist_ok=True)

def guardar_csv(registros, nombre):
    """Guarda una lista de dicts como CSV y reporta el número de registros."""
    df = pd.DataFrame(registros)
    ruta = os.path.join(OUT_DIR, nombre)
    df.to_csv(ruta, index=False, encoding="utf-8")
    print(f"{nombre}: {len(df)} registros -> {ruta}")
    return df

print("Configuración lista.")

Configuración lista.


## 3.1 API 1 — CountriesNow (demografía y geografía)

- **Endpoint:** `https://countriesnow.space/api/v0.1/countries/population`
- **Autenticación:** ninguna (API abierta, sin key).
- **Registros objetivo:** ≥ 200 (una llamada devuelve ~240 países).
- **Paginación:** no es necesaria; una sola llamada trae todos los países.
- **Salida:** `dataset_api_1.csv`

In [5]:
# =============================================================================
# API 1 — CountriesNow: población de los países (demografía)
# -----------------------------------------------------------------------------
# LÓGICA GENERAL:
#   1) Se consulta el endpoint /countries/population, que devuelve TODOS los
#      países con su historial de población. Es una API abierta (sin key).
#   2) Se valida el PAYLOAD (campo 'error' y lista 'data'), no solo el status,
#      para no dar por buena una respuesta de error con código 200.
#   3) Cada país trae 'populationCounts' (lista por año); se APLANA tomando el
#      año más reciente -> una fila por país.
#   4) Una sola llamada supera los 200 registros, así que NO requiere paginación.
#   5) Se arma el DataFrame y se guarda como dataset_api_1.csv.
# =============================================================================

# --- Constantes de la fuente ---
CN_URL = "https://countriesnow.space/api/v0.1/countries/population"


def descargar_poblacion(reintentos=3, espera=5):
    """
    Descarga los datos de población de todos los países desde CountriesNow.
    Devuelve: la lista 'data' (países), o [] si falla o el payload es inválido.
    """
    for intento in range(1, reintentos + 1):
        try:
            resp = requests.get(CN_URL,
                                headers={"User-Agent": "TallerMLY1101-countriesnow/1.0"},
                                timeout=60)
            if resp.status_code == 429:
                print(f"   [429] límite de tasa; esperando {espera*intento}s...")
                time.sleep(espera * intento)
                continue
            resp.raise_for_status()
            data = resp.json()
            # Validación de contenido: 'error' debe ser False y 'data' una lista.
            if isinstance(data, dict) and data.get("error") is False and isinstance(data.get("data"), list):
                return data["data"]
            print("   payload inesperado:", str(data)[:200])
            return []
        except requests.RequestException as e:
            print(f"   Error de conexión ({type(e).__name__}); reintento {intento}/{reintentos}")
            time.sleep(espera * intento)
    return []


def aplanar_poblacion(item):
    """
    Convierte un país (con historial de población) en una fila:
    toma el registro del AÑO MÁS RECIENTE de 'populationCounts'.
    """
    counts = item.get("populationCounts") or []
    # max por año; si no hay conteos, dict vacío (poblacion/anio quedarán en None).
    ultimo = max(counts, key=lambda c: int(c.get("year", 0))) if counts else {}
    return {
        "pais":      item.get("country"),
        "code":      item.get("code"),      # ISO2
        "iso3":      item.get("iso3"),      # ISO3
        "poblacion": ultimo.get("value"),
        "anio":      ultimo.get("year"),
    }


# --- 1) Descargar ---
paises = descargar_poblacion()
print(f"Países recibidos: {len(paises)}")

# --- 2) Aplanar cada país a una fila ---
registros = [aplanar_poblacion(p) for p in paises]

# --- 3) DataFrame + guardado (con resguardo si vino vacío) ---
df_api1 = pd.DataFrame(registros)

if len(df_api1) == 0:
    print("\nADVERTENCIA: CountriesNow no devolvió datos en esta corrida. "
        "No se sobrescribe dataset_api_1.csv; reintenta esta celda en unos minutos.")
else:
    guardar_csv(registros, "dataset_api_1.csv")
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", None)
    print(f"\nRegistros totales: {len(df_api1)}")
    print(f"Columnas ({len(df_api1.columns)}): {list(df_api1.columns)}")

df_api1   # última expresión: muestra la tabla en pantalla

Países recibidos: 263
dataset_api_1.csv: 263 registros -> .\dataset_api_1.csv

Registros totales: 263
Columnas (5): ['pais', 'code', 'iso3', 'poblacion', 'anio']


,pais,code,iso3,poblacion,anio
0,Arab World,ARB,ARB,419790588,2018
1,Caribbean small states,CSS,CSS,7358965,2018
2,Central Europe and the Baltics,CEB,CEB,102511922,2018
3,Early-demographic dividend,EAR,EAR,3249140605,2018
4,East Asia & Pacific,EAS,EAS,2328220870,2018
...,...,...,...,...,...
258,Virgin Islands (U.S.),VIR,VIR,106977,2018
259,West Bank and Gaza,PSE,PSE,4569087,2018
260,"Yemen, Rep.",YEM,YEM,28498687,2018
261,Zambia,ZMB,ZMB,17351822,2018


## 3.2 API 2 — World Bank (economía)

- **Endpoint:** `https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD?format=json`
- **Autenticación:** ninguna.
- **Registros objetivo:** ≥ 200.
- **Paginación:** la API pagina los resultados (`page` / `per_page`); se recorren las páginas y se acumulan → aquí se implementa la paginación.
- **Salida:** `dataset_api_2.csv`

In [ ]:
# Descarga desde World Bank -> dataset_api_2.csv

## 3.3 API 3 — Nager.Date (cultura: días festivos)

- **Endpoints:** `.../v3/AvailableCountries` y `.../v3/PublicHolidays/{año}/{país}`
- **Autenticación:** ninguna, sin límite de tasa.
- **Registros objetivo:** ≥ 200 (se recorren los países disponibles y se acumulan sus feriados).
- **Salida:** `dataset_api_3.csv`

In [ ]:
# Descarga desde Nager.Date -> dataset_api_3.csv